# Transfer Learning on Oxford Flowers 102 Dataset

Applying transfer learning techniques using pre-trained convolutional neural networks (ResNet50, VGG16, and MobileNetV2) to classify images from the Oxford Flowers 102 dataset. Compare the performance of the different models on this dataset.

**Goal**
Apply transfer learning for flower classification using the Oxford Flowers 102 dataset.

**Pre-trained models to be used**
RestNet50, VGG16, MobileNetV2, Inception(GoogleNet)

**Intro to Oxford Flowers 102 dataset**
It consists of 102 flower categories. The flowers chosen to be flower commonly occuring in the United Kingdom. Each class consists of between 40 and 258 images.

## Data Loading & Exploration

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet50
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inceptionv3
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical

In [ ]:
# Load the Oxford Flowers 102 dataset
dataset, info = tfds.load('oxford_flowers102:2.1.1', with_info=True, as_supervised=True)

# Split dataset into training, validation, and testing sets
train_dataset = dataset['train']
validation_dataset = dataset['validation']
test_dataset = dataset['test']

# Get the number of classes (should be 102)
num_classes = info.features['label'].num_classes

In [ ]:
example = next(iter(train_dataset))
image, label = example
plt.imshow(image)
plt.title(f"Label: {label.numpy()}")
plt.axis('off')
plt.show()

In [ ]:
print(f"Image shape: {image.shape}")

## Data Preprocessing:

1. Load the Oxford Flowers 102 Dataset, then preprocess the images for each model (resizing, applying the correct preprocessing function, etc.), and finally prepare the data for training.

2. Apply Image Preprocessing, images need to be resized to match the input dimensions for each model and preprocessed with the appropriate preprocessing function.

3. One-hot Encoding the Labels:
Since we're using categorical cross-entropy loss for multi-class classification, we'll one-hot encode the labels.

4. Batching and Prefetching:
We'll batch and prefetch the dataset for efficiency.

In [ ]:
# Resize and preprocess images for each model
def preprocess_images(image, label, model_type='resnet50'):
    if model_type == 'resnet50' or model_type == 'vgg16' or model_type == 'mobilenetv2':
        # Resize to 224x224 for ResNet50, VGG16, and MobileNetV2
        image = tf.image.resize(image, (224, 224))
    elif model_type == 'inceptionv3':
        # Resize to 299x299 for InceptionV3 (GoogleNet)
        image = tf.image.resize(image, (299, 299))

    if model_type == 'resnet50':
        image = preprocess_resnet50(image)
    elif model_type == 'vgg16':
        image = preprocess_vgg16(image)
    elif model_type == 'mobilenetv2':
        image = preprocess_mobilenetv2(image)
    elif model_type == 'inceptionv3':
        image = preprocess_inceptionv3(image)
    return image, label

In [ ]:
# Modify preprocessing to one-hot encode the labels
def preprocess_images_and_labels(image, label, model_type='resnet50'):
    # Preprocess the image (resize and apply model-specific preprocessing)
    image, label = preprocess_images(image, label, model_type)

    # One-hot encode the label
    label = to_categorical(label, num_classes=num_classes)

    return image, label

# Update the dataset preprocessing to apply one-hot encoding
train_dataset_resnet50 = train_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='resnet50'))
validation_dataset_resnet50 = validation_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='resnet50'))
test_dataset_resnet50 = test_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='resnet50'))

train_dataset_vgg16 = train_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='vgg16'))
validation_dataset_vgg16 = validation_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='vgg16'))
test_dataset_vgg16 = test_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='vgg16'))

train_dataset_mobilenetv2 = train_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='mobilenetv2'))
validation_dataset_mobilenetv2 = validation_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='mobilenetv2'))
test_dataset_mobilenetv2 = test_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='mobilenetv2'))

train_dataset_inceptionv3 = train_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='inceptionv3'))
validation_dataset_inceptionv3 = validation_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='inceptionv3'))
test_dataset_inceptionv3 = test_dataset.map(lambda image, label: preprocess_images_and_labels(image, label, model_type='inceptionv3'))

In [ ]:
# Batch and Prefetch for efficiency
train_dataset_resnet50 = train_dataset_resnet50.batch(32).prefetch(tf.data.experimental.AUTOTUNE)
validation_dataset_resnet50 = validation_dataset_resnet50.batch(32).prefetch(tf.data.experimental.AUTOTUNE)
test_dataset_resnet50 = test_dataset_resnet50.batch(32).prefetch(tf.data.experimental.AUTOTUNE)

train_dataset_vgg16 = train_dataset_vgg16.batch(32).prefetch(tf.data.experimental.AUTOTUNE)
validation_dataset_vgg16 = validation_dataset_vgg16.batch(32).prefetch(tf.data.experimental.AUTOTUNE)
test_dataset_vgg16 = test_dataset_vgg16.batch(32).prefetch(tf.data.experimental.AUTOTUNE)

train_dataset_mobilenetv2 = train_dataset_mobilenetv2.batch(32).prefetch(tf.data.experimental.AUTOTUNE)
validation_dataset_mobilenetv2 = validation_dataset_mobilenetv2.batch(32).prefetch(tf.data.experimental.AUTOTUNE)
test_dataset_mobilenetv2 = test_dataset_mobilenetv2.batch(32).prefetch(tf.data.experimental.AUTOTUNE)

train_dataset_inceptionv3 = train_dataset_inceptionv3.batch(32).prefetch(tf.data.experimental.AUTOTUNE)
validation_dataset_inceptionv3 = validation_dataset_inceptionv3.batch(32).prefetch(tf.data.experimental.AUTOTUNE)
test_dataset_inceptionv3 = test_dataset_inceptionv3.batch(32).prefetch(tf.data.experimental.AUTOTUNE)

## Model Adaptation and Training:

Now, let's adapt the code to load pre-trained models (ResNet50, VGG16, MobileNetV2, and InceptionV3), add custom layers on top, and train the models.

### ResNet50

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

base_model_resnet50 = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

x = GlobalAveragePooling2D()(base_model_resnet50.output)
x = Dense(1024, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)

model_resnet50 = Model(inputs=base_model_resnet50.input, outputs=predictions)

model_resnet50.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

### VGG16

In [ ]:
from tensorflow.keras.applications import VGG16

base_model_vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

x = GlobalAveragePooling2D()(base_model_vgg16.output)
x = Dense(512, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)

model_vgg16 = Model(inputs=base_model_vgg16.input, outputs=predictions)

model_vgg16.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


### MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2

base_model_mobilenetv2 = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

x = GlobalAveragePooling2D()(base_model_mobilenetv2.output)
x = Dense(256, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)

model_mobilenetv2 = Model(inputs=base_model_mobilenetv2.input, outputs=predictions)

model_mobilenetv2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

### InceptionV3 (GoogleNet)

In [ ]:
from tensorflow.keras.applications import InceptionV3

base_model_inceptionv3 = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

x = GlobalAveragePooling2D()(base_model_inceptionv3.output)
x = Dense(512, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)

model_inceptionv3 = Model(inputs=base_model_inceptionv3.input, outputs=predictions)

model_inceptionv3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## Model Training

In [ ]:
epochs = 4

# Train ResNet50
history_resnet50 = model_resnet50.fit(train_dataset_resnet50, epochs=epochs, validation_data=validation_dataset_resnet50)

# Train VGG16
history_vgg16 = model_vgg16.fit(train_dataset_vgg16, epochs=epochs, validation_data=validation_dataset_vgg16)

# Train MobileNetV2
history_mobilenetv2 = model_mobilenetv2.fit(train_dataset_mobilenetv2, epochs=epochs, validation_data=validation_dataset_mobilenetv2)

# Train InceptionV3
history_inceptionv3 = model_inceptionv3.fit(train_dataset_inceptionv3, epochs=epochs, validation_data=validation_dataset_inceptionv3)

## Model Evaluation

In [ ]:
# Evaluate models on test dataset
acc_resnet50 = model_resnet50.evaluate(test_dataset_resnet50)[1]
acc_vgg16 = model_vgg16.evaluate(test_dataset_vgg16)[1]
acc_mobilenetv2 = model_mobilenetv2.evaluate(test_dataset_mobilenetv2)[1]
acc_inceptionv3 = model_inceptionv3.evaluate(test_dataset_inceptionv3)[1]

print(f'ResNet50 Accuracy: {acc_resnet50:.2f}')
print(f'VGG16 Accuracy: {acc_vgg16:.2f}')
print(f'MobileNetV2 Accuracy: {acc_mobilenetv2:.2f}')
print(f'InceptionV3 Accuracy: {acc_inceptionv3:.2f}')

## Plotting Training History

In [ ]:
# Function to plot the training history
def plot_history(history, model_name):
    plt.figure(figsize=(12, 4))

    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{model_name} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{model_name} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Plot history for each model
plot_history(history_resnet50, 'ResNet50')
plot_history(history_vgg16, 'VGG16')
plot_history(history_mobilenetv2, 'MobileNetV2')
plot_history(history_inceptionv3, 'InceptionV3')

## Conclusion

ResNet50 achieved an accuracy of 1% on the test set, with training accuracy improving significantly but with a large gap between training and validation performance.

VGG16 showed similar performance, with a final accuracy of 1%.

MobileNetV2 achieved a slightly better performance with a test accuracy of 6%.

InceptionV3 performed the best among the four models with a test accuracy of 8%.